# DEX–CEX Market Analysis: USDC Peg Deviation

**Task 2** — USDC Peg Deviation: DEX vs CEX (Hourly)

| Parameter | Value |
|-----------|-------|
| Timeframe | 2025-07-01 to 2025-09-30 (UTC) |
| DEX | Uniswap v3 USDC/USDT 0.01% pool |
| CEX | Bybit USDC/USDT spot |
| Band | ±0.1% around 1.0000 (0.9990 – 1.0010) |

**Data Sources:**
- **Uniswap**: The Graph Protocol — Uniswap v3 Subgraph (swap-level data)
- **Bybit**: Public REST API `/v5/market/kline` (hourly OHLCV candles)

In [26]:
import aiohttp
import pandas as pd
import nest_asyncio
nest_asyncio.apply()

# The Graph
GRAPH_API_KEY = '5f777368d453e1fa2e835f09e265d4f6'
SUBGRAPH_ID = "5zvR82QoaXYFyDEKLZ9t6v9adgnptxYpKpSbxtgVENFV"
GRAPHQL_ENDPOINT = f"https://gateway.thegraph.com/api/{GRAPH_API_KEY}/subgraphs/id/{SUBGRAPH_ID}"

# Uniswap v3 USDC/USDT 0.01% pool
POOL_ADDRESS = "0x3416cf6c708da44db2624d63ea0aaef7113527c6"

# Bybit
BYBIT_URL = "https://api.bybit.com"

# Time period
START_DATE = "2025-07-01"
END_DATE = "2025-09-30"

# Peg band
PEG_CENTER = 1.0000
BAND_LOWER = 0.9990
BAND_UPPER = 1.0010

## 1. Data Fetchers

### Uniswap v3 — Swap-level data via The Graph

**Pool**: USDC/USDT 0.01% fee tier  
**Address**: `0x3416cf6c708da44db2624d63ea0aaef7113527c6`  
**Network**: Ethereum Mainnet

In [27]:
async def fetch_pool_swaps(pool_address, start_date, end_date):
    all_swaps = []
    last_id = ""
    cursor = int(pd.Timestamp(start_date, tz="UTC").timestamp())
    end_ts = int(pd.Timestamp(end_date, tz="UTC").timestamp()) + 86400

    while cursor < end_ts:
        id_filter = f'id_gt: "{last_id}"' if last_id else ""
        query = f"""
            query {{
              swaps(
                first: 1000
                where: {{
                  pool: "{pool_address.lower()}"
                  timestamp_gte: {cursor}
                  timestamp_lte: {end_ts}
                  {id_filter}
                }}
                orderBy: timestamp
                orderDirection: asc
              ) {{
                id timestamp amount0 amount1 sqrtPriceX96
              }}
            }}
        """
        async with aiohttp.ClientSession() as session:
            async with session.post(GRAPHQL_ENDPOINT, json={'query': query}) as resp:
                if resp.status != 200:
                    print(f"GraphQL error: status {resp.status}")
                    return None
                result = await resp.json()
                swaps = result.get('data', {}).get('swaps', [])

        if not swaps:
            break

        all_swaps.extend(swaps)
        print(f"\r  {len(all_swaps)} swaps fetched", end="", flush=True)
        last_id = swaps[-1]['id']
        cursor = int(swaps[-1]['timestamp'])
        await nest_asyncio.asyncio.sleep(0.3)

    print(f"\nDone \u2014 {len(all_swaps)} total swaps.")
    return all_swaps

### Bybit CEX — Hourly OHLCV candles

**Pair**: USDC/USDT spot  
**Endpoint**: `/v5/market/kline` (public, no auth required)

In [28]:
async def fetch_bybit_klines(symbol, start_date, end_date,
                              category="spot", interval="60"):
    all_candles = []
    start_ms = int(pd.Timestamp(start_date, tz="UTC").timestamp() * 1000)
    end_ms = int((pd.Timestamp(end_date, tz="UTC").timestamp() + 86400) * 1000)
    current_end = end_ms

    while current_end > start_ms:
        params = (
            f"category={category}&end={current_end}"
            f"&interval={interval}&limit=1000"
            f"&start={start_ms}&symbol={symbol}"
        )
        full_url = f"{BYBIT_URL}/v5/market/kline?{params}"

        async with aiohttp.ClientSession() as session:
            async with session.get(full_url) as resp:
                import json as _json
                raw = await resp.text()
                data = _json.loads(raw) if raw else None

        if data is None or data.get("retCode", -1) != 0:
            print(f"  API error: {data}")
            break

        candles = data.get("result", {}).get("list", [])
        if not candles:
            break

        all_candles.extend(candles)
        oldest_ts = int(candles[-1][0])
        if oldest_ts >= current_end:
            break
        current_end = oldest_ts

        print(f"\r  {len(all_candles)} candles fetched", end="", flush=True)
        await nest_asyncio.asyncio.sleep(0.2)

    print(f"\nDone \u2014 {len(all_candles)} total candles.")

    if not all_candles:
        return pd.DataFrame()

    df = pd.DataFrame(all_candles, columns=[
        "startTime", "open", "high", "low", "close", "volume", "turnover"
    ])
    df["startTime"] = pd.to_numeric(df["startTime"])
    df["datetime"] = pd.to_datetime(df["startTime"], unit="ms", utc=True)
    for col in ["open", "high", "low", "close", "volume", "turnover"]:
        df[col] = df[col].astype(float)
    df = df.drop_duplicates(subset="startTime").sort_values("datetime").reset_index(drop=True)
    return df

## 2. Fetch Raw Data

In [29]:
# Fetch Uniswap swap-level data
raw_swaps = nest_asyncio.asyncio.run(
    fetch_pool_swaps(POOL_ADDRESS, START_DATE, END_DATE)
)

usdc_usdt_swaps_df = (
    pd.DataFrame(raw_swaps)
    .assign(
        datetime=lambda df: pd.to_datetime(df['timestamp'].astype(int), unit='s', utc=True),
        amount0=lambda df: df['amount0'].astype(float),
        amount1=lambda df: df['amount1'].astype(float),
    )
    .drop(columns=['id', 'timestamp'])
    .sort_values('datetime')
    .reset_index(drop=True)
)
usdc_usdt_swaps_df['price'] = (usdc_usdt_swaps_df['amount1'] / usdc_usdt_swaps_df['amount0']).abs()

print(f"Total swaps: {len(usdc_usdt_swaps_df)}")
print(f"Date range:  {usdc_usdt_swaps_df['datetime'].min()} to {usdc_usdt_swaps_df['datetime'].max()}")
usdc_usdt_swaps_df

  8007 swaps fetched
Done — 8007 total swaps.
Total swaps: 8007
Date range:  2025-07-01 00:01:35+00:00 to 2025-09-29 14:22:11+00:00


,amount0,amount1,sqrtPriceX96,datetime,price
0,5807.142477,-5804.943048,79217113797656520221235955071,2025-07-01 00:01:35+00:00,0.999621
1,85.223595,-85.191306,79217113662578375686596692604,2025-07-01 00:02:11+00:00,0.999621
2,-32.360874,32.355085,79217113713875015287136604890,2025-07-01 00:03:11+00:00,0.999821
3,192.702829,-192.629819,79217113408443865901405003124,2025-07-01 00:03:47+00:00,0.999621
4,-299.026137,298.972639,79217113882443297781621384533,2025-07-01 00:05:11+00:00,0.999821
...,...,...,...,...,...
8002,34.432197,-34.421888,79220263528277692276067809158,2025-08-28 04:42:59+00:00,0.999701
8003,105.000000,-104.973865,79222263213567718225068265493,2025-09-01 03:35:11+00:00,0.999751
8004,1888.252284,-1887.610786,79218662786736455836852609509,2025-09-07 11:21:47+00:00,0.999660
8005,-2911.598374,2911.254884,79219530214068814421991963014,2025-09-10 11:39:47+00:00,0.999882


In [30]:
# Fetch Bybit hourly candles
bybit_df = nest_asyncio.asyncio.run(
    fetch_bybit_klines("USDCUSDT", START_DATE, END_DATE, category="spot", interval="60")
)
bybit_df['price'] = bybit_df['turnover'] / bybit_df['volume']

print(f"Total candles: {len(bybit_df)}")
print(f"Date range:    {bybit_df['datetime'].min()} to {bybit_df['datetime'].max()}")
bybit_df

  2211 candles fetched
Done — 2211 total candles.
Total candles: 2209
Date range:    2025-07-01 00:00:00+00:00 to 2025-10-01 00:00:00+00:00


,startTime,open,high,low,close,volume,turnover,datetime,price
0,1751328000000,0.9998,0.9998,0.9997,0.9998,655542.98,6.553723e+05,2025-07-01 00:00:00+00:00,0.999740
1,1751331600000,0.9998,0.9998,0.9997,0.9998,1362633.10,1.362245e+06,2025-07-01 01:00:00+00:00,0.999715
2,1751335200000,0.9998,0.9998,0.9997,0.9997,502373.52,5.022497e+05,2025-07-01 02:00:00+00:00,0.999754
3,1751338800000,0.9997,0.9998,0.9997,0.9997,2619473.87,2.618720e+06,2025-07-01 03:00:00+00:00,0.999712
4,1751342400000,0.9997,0.9998,0.9997,0.9997,1522100.07,1.521662e+06,2025-07-01 04:00:00+00:00,0.999712
...,...,...,...,...,...,...,...,...,...
2204,1759262400000,0.9997,0.9997,0.9996,0.9996,7769678.21,7.767134e+06,2025-09-30 20:00:00+00:00,0.999673
2205,1759266000000,0.9996,0.9997,0.9996,0.9997,3161509.48,3.160345e+06,2025-09-30 21:00:00+00:00,0.999632
2206,1759269600000,0.9997,0.9997,0.9996,0.9996,2692616.35,2.691652e+06,2025-09-30 22:00:00+00:00,0.999642
2207,1759273200000,0.9996,0.9997,0.9996,0.9996,1512567.51,1.512047e+06,2025-09-30 23:00:00+00:00,0.999656


## 3. Build Unified Hourly Comparison Table

One row per hour. For each venue, we compute:
- **volume**: total USDC volume traded outside the ±0.1% band
- **min/max price**: extremes of outside-band trades that hour

> **Note on Bybit data**: Bybit provides pre-aggregated hourly OHLCV candles, so we cannot
> isolate individual trades outside the band. We flag an hour as "outside-band" when the
> candle's low < 0.9990 or high > 1.0010, and report the full candle volume as an
> upper-bound estimate. Uniswap swap-level data allows exact filtering.

In [31]:
# All hours in the timeframe
all_hours = pd.date_range(START_DATE, END_DATE, freq='h', tz='UTC', inclusive='both')

# --- Uniswap: outside-band swaps aggregated hourly ---
uni_outside = usdc_usdt_swaps_df[
    (usdc_usdt_swaps_df['price'] < BAND_LOWER) |
    (usdc_usdt_swaps_df['price'] > BAND_UPPER)
]
uniswap_hourly = (
    uni_outside
    .groupby(pd.Grouper(key='datetime', freq='h'))
    .agg(
        uniswap_volume=('amount0', lambda x: x.abs().sum()),
        uniswap_min_price=('price', 'min'),
        uniswap_max_price=('price', 'max'),
    )
)

# --- Bybit: outside-band approximation from hourly candles ---
bybit_outside = bybit_df[
    (bybit_df['low'] < BAND_LOWER) |
    (bybit_df['high'] > BAND_UPPER)
]
bybit_hourly = (
    bybit_outside
    .set_index('datetime')
    [['volume', 'low', 'high']]
    .rename(columns={
        'volume': 'bybit_volume',
        'low': 'bybit_min_price',
        'high': 'bybit_max_price',
    })
)

# --- Build unified DataFrame: one row per hour ---
unified_df = (
    pd.DataFrame(index=all_hours)
    .rename_axis('time')
    .join(uniswap_hourly)
    .join(bybit_hourly)
    .reset_index()
)

# Fill missing volumes with 0 (no outside-band activity that hour)
unified_df['uniswap_volume'] = unified_df['uniswap_volume'].fillna(0)
unified_df['bybit_volume'] = unified_df['bybit_volume'].fillna(0)

# Ensure column order matches the challenge spec
unified_df = unified_df[[
    'time', 'uniswap_volume', 'bybit_volume',
    'uniswap_min_price', 'uniswap_max_price',
    'bybit_min_price', 'bybit_max_price'
]]

print(f"Shape: {unified_df.shape}")
print(f"Hours with Uniswap outside-band activity: {(unified_df['uniswap_volume'] > 0).sum()}")
print(f"Hours with Bybit outside-band activity:   {(unified_df['bybit_volume'] > 0).sum()}")
print(f"Hours with BOTH venues outside-band:      "
      f"{((unified_df['uniswap_volume'] > 0) & (unified_df['bybit_volume'] > 0)).sum()}")
unified_df

Shape: (2185, 7)
Hours with Uniswap outside-band activity: 4
Hours with Bybit outside-band activity:   6
Hours with BOTH venues outside-band:      0


,time,uniswap_volume,bybit_volume,uniswap_min_price,uniswap_max_price,bybit_min_price,bybit_max_price
0,2025-07-01 00:00:00+00:00,0.0,0.0,NaN,NaN,NaN,NaN
1,2025-07-01 01:00:00+00:00,0.0,0.0,NaN,NaN,NaN,NaN
2,2025-07-01 02:00:00+00:00,0.0,0.0,NaN,NaN,NaN,NaN
3,2025-07-01 03:00:00+00:00,0.0,0.0,NaN,NaN,NaN,NaN
4,2025-07-01 04:00:00+00:00,0.0,0.0,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...
2180,2025-09-29 20:00:00+00:00,0.0,0.0,NaN,NaN,NaN,NaN
2181,2025-09-29 21:00:00+00:00,0.0,0.0,NaN,NaN,NaN,NaN
2182,2025-09-29 22:00:00+00:00,0.0,0.0,NaN,NaN,NaN,NaN
2183,2025-09-29 23:00:00+00:00,0.0,0.0,NaN,NaN,NaN,NaN


## 4. Export CSV

In [32]:
unified_df.to_csv('usdc_peg_deviation_hourly.csv', index=False)
print(f"Exported {len(unified_df)} rows to usdc_peg_deviation_hourly.csv")

Exported 2185 rows to usdc_peg_deviation_hourly.csv
